### Tools In Langchain

1. How to Create Tools
2. How to use Built In Tools and toolkits
3. How to use chat models to call tools
4. How to pass tool outputs to chat models

In [1]:
## @tool decorator (Simple way to create a tool)

from langchain_core.tools import tool

@tool
def division(a:int, b:int)-> int:
    """Divide 2 numbers"""
    return a/b


In [3]:
print(division.name)
print(division.description)
print(division.args)

division
Divide 2 numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [7]:
## async implementation (It will run parallely with the program)

from langchain_core.tools import tool

@tool
async def amultiply(a:int, b:int)-> int:
    """"Multiply two numbers"""
    return a * b

In [5]:
from typing import Annotated, List

## Annotated -> It takes the type and description for a variable

@tool
def multiply_by_max(a: Annotated[int, "A value"], b: Annotated[List[int], "list of int"])->int:
    """Multiply a by max of b"""
    return a * max(b)

In [6]:
print(multiply_by_max.args)

{'a': {'description': 'A value', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'list of int', 'items': {'type': 'integer'}, 'title': 'B', 'type': 'array'}}


### Structured Tool
A Python function that an LLM can call using clean, structured inputs (JSON-like) instead of messy text.

In [10]:
from langchain_core.tools import StructuredTool

def multiply(a:int, b:int)-> int:
    """"Multiply two numbers"""
    return a * b

async def amultiply(a:int, b:int)-> int:
    """"Multiply two numbers"""
    return a * b

calculator = StructuredTool.from_function(func=multiply, coroutine=amultiply)

print(calculator.invoke({"a": 2, "b": 3})) # multiply
print(await calculator.ainvoke({"a": 2, "b": 5})) # amultiply

6
10


### In-built Tools

### Wikipedia Integration

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=5, doc_content_chars_max=500)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(tool.invoke({"query": "Langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Vector database
Summary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in v


In [ ]:
## Arxiv Tool
from langchain_community.utilities import ArxivAPIWrapper

tool = ArxivAPIWrapper()

tool.run("Attention is all you need")

'Published: 2021-05-06\nTitle: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet\nAuthors: Luke Melas-Kyriazi\nSummary: The strong performance of vision transformers on image classification and other vision tasks is often attributed to the design of their multi-head attention layers. However, the extent to which attention is responsible for this strong performance remains unclear. In this short report, we ask: is the attention layer even necessary? Specifically, we replace the attention layer in a vision transformer with a feed-forward layer applied over the patch dimension. The resulting architecture is simply a series of feed-forward layers applied over the patch and feature dimensions in an alternating fashion. In experiments on ImageNet, this architecture performs surprisingly well: a ViT/DeiT-base-sized model obtains 74.9\\% top-1 accuracy, compared to 77.9\\% and 79.9\\% for ViT and DeiT respectively. These results indicate that aspects

### Call tool with LLM Model

In [16]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=100)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(wiki_tool.invoke({"query": "Langchain"}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of 


In [17]:
from langchain_core.tools import tool

@tool 
def add(a: int, b: int)->int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Mulitply two numbers"""
    return a * b

In [30]:
wiki_tool.name

'wikipedia'

In [31]:
add.name

'add'

In [32]:
multiply.name

'multiply'

In [18]:
tools = [wiki_tool, add, multiply]

In [19]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\C\\GenAI and Agentic AI\\Langchain\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=100)),
 StructuredTool(name='add', description='Add two numbers', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x0000025A2C554180>),
 StructuredTool(name='multiply', description='Mulitply two numbers', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x0000025A2C554360>)]

In [21]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [23]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [24]:
from langchain.chat_models import init_chat_model
llm = init_chat_model("qwen/qwen3-32b", model_provider="groq")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025A2C78FB60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025A2CF2C980>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [25]:
llm_with_tools = llm.bind_tools(tools)
llm_with_tools

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025A2C78FB60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025A2CF2C980>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'wikipedia', 'description': 'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.', 'parameters': {'properties': {'query': {'description': 'query to look up on wikipedia', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function

In [35]:
from langchain_core.messages import HumanMessage
query = "What is 2 * 3"
messages = [HumanMessage(query)]

response = llm_with_tools.invoke(query)
print(response) # response.content will be empty bc tool ['multiply'] gets called as reasoning_content

content='' additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user is asking "What is 2 * 3". I need to figure out which tool to use here. Looking at the available functions, there\'s one called multiply that takes two integers. The parameters are a and b, both integers. So 2 multiplied by 3 would use the multiply function with a=2 and b=3. I should call that function and return the result.\n', 'tool_calls': [{'id': 'q2tq9td5w', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 116, 'prompt_tokens': 324, 'total_tokens': 440, 'completion_time': 0.180296233, 'completion_tokens_details': {'reasoning_tokens': 87}, 'prompt_time': 0.01521376, 'prompt_tokens_details': None, 'queue_time': 0.052497879, 'total_time': 0.195509993}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider'

In [ ]:
response ## It reads the docs string of the function too.

AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user is asking "What is 2 * 3". I need to figure out which tool to use here. Looking at the available functions, there\'s one called multiply. The description says it\'s for multiplying two numbers. The parameters are a and b, both integers. So, in this case, 2 and 3 are the numbers. I should call the multiply function with a=2 and b=3. That should give the answer 6. Let me make sure I\'m not mixing up the functions. The add function is for addition, so that\'s not it here. Yep, multiply is the right choice.\n', 'tool_calls': [{'id': 'x6edjv1ja', 'function': {'arguments': '{"a":2,"b":3}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 163, 'prompt_tokens': 324, 'total_tokens': 487, 'completion_time': 0.302751539, 'completion_tokens_details': {'reasoning_tokens': 134}, 'prompt_time': 0.012957995, 'prompt_tokens_details': None, 'queue_time': 0.30351291

In [28]:
response.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': 'x6edjv1ja',
  'type': 'tool_call'}]

In [36]:

for tool_call in response.tool_calls:
    selected_tool = {"add": add, "multiply": multiply, "wikipedia": wiki_tool}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

[HumanMessage(content='What is 2 * 3', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='6', name='multiply', tool_call_id='q2tq9td5w')]

In [37]:
# Now, we have to send response back to llm
llm_with_tools.invoke(messages)

AIMessage(content='The result of multiplying 2 by 3 is **6**.\n\n$$\n2 \\times 3 = 6\n$$', additional_kwargs={'reasoning_content': 'Okay, the user asked "What is 2 * 3". I need to figure out the right tool to use here. Let me check the available functions. There\'s the multiply function which takes two integers. Perfect, since 2 and 3 are both numbers. I\'ll call the multiply function with a=2 and b=3. The response came back as 6, so I just need to present that clearly. The answer is 6.\n'}, response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 334, 'total_tokens': 458, 'completion_time': 0.245628334, 'completion_tokens_details': {'reasoning_tokens': 93}, 'prompt_time': 0.014059506, 'prompt_tokens_details': None, 'queue_time': 0.051861587, 'total_time': 0.25968784}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dce26-f03e-799

In [47]:
from langchain_core.messages import HumanMessage
query = "What is Langchain and what is 5 * 12"
messages = [HumanMessage(query)]

ai_msg = llm_with_tools.invoke(query)
ai_msg

AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking two things here: what Langchain is and the result of 5 multiplied by 12. Let me break this down.\n\nFirst, for the definition of Langchain. I need to use the Wikipedia tool to get an accurate description. The user probably wants a concise explanation of what Langchain does. So I\'ll call the Wikipedia function with the query "Langchain" to fetch the relevant information.\n\nNext, the math problem: 5 * 12. This is a straightforward multiplication. Looking at the available tools, there\'s a multiply function that takes two integers. So I\'ll use that function with a=5 and b=12. It\'s better to use the tool here since it\'s a simple calculation and the function exists for this purpose.\n\nI should handle both parts separately. First, call Wikipedia for Langchain, then call the multiply function. The user might be working on a project that involves both understanding Langchain and performing some calcul

In [48]:
ai_msg.tool_calls

[{'name': 'wikipedia',
  'args': {'query': 'Langchain'},
  'id': 'rjwj7yx80',
  'type': 'tool_call'},
 {'name': 'multiply',
  'args': {'a': 5, 'b': 12},
  'id': 'fc648qp55',
  'type': 'tool_call'}]

In [51]:
for tool_call in ai_msg.tool_calls:
    selected_tool = {"add": add, "multiply": multiply, "wikipedia": wiki_tool}[tool_call["name"].lower()]
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

[HumanMessage(content='What is Langchain and what is 5 * 12', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='6', name='multiply', tool_call_id='q2tq9td5w'),
 ToolMessage(content='Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of ', name='wikipedia', tool_call_id='rjwj7yx80'),
 ToolMessage(content='60', name='multiply', tool_call_id='fc648qp55')]

In [53]:
# Now, we have to send response back to llm
response = llm_with_tools.invoke(messages)

In [54]:
response.content

'LangChain is a software framework designed to help developers integrate and connect various machine learning models, data sources, and applications. It provides tools for managing prompts, handling data transformations, and orchestrating workflows, making it easier to build end-to-end solutions with AI models. \n\nAdditionally, $5 \\times 12 = 60$.'